# Bird's Eye View Of the Notebook

This notebook is a cleaned, commented version of the original workflow.

It performs the following steps in order:

1. Load the LendingClub, Treasury bill, and Fed Funds datasets.
2. Clean the LendingClub loan-level data.
3. Aggregate LendingClub data to **monthly** frequency.
4. Prepare the macroeconomic series at the same monthly frequency.
5. Merge everything into one modeling table.
6. Verify that the monthly timeline is continuous.
7. Create a **time-based 75/25 train-test split** to avoid leakage.
8. Save the final datasets as `train.csv` and `test.csv`.

The logic matches the original notebook, but the code is organized to be easier to read and maintain.


In [1]:
import pandas as pd

# Display configuration for easier inspection inside the notebook.
pd.set_option("display.max_columns", None)

## 1. File paths and data loading




In [ ]:
# Input files
# Download the data from https://www.kaggle.com/datasets/wordsforthewise/lending-club  and save it as LendingClub.csv in the Data folder
LENDINGCLUB_FILE = "../Data/LendingClub.csv"
FEDFUNDS_FILE = "../Data/FEDFUNDS.csv"
TBILL_FILE = "../Data/tbill.csv"

# Read source data
lendingclub = pd.read_csv(
    LENDINGCLUB_FILE,
    usecols=["id", "issue_d", "loan_amnt", "term", "int_rate"],
)

fed_data = pd.read_csv(FEDFUNDS_FILE)
tbill = pd.read_csv(TBILL_FILE)

print("LendingClub shape:", lendingclub.shape)
print("FED data shape:   ", fed_data.shape)
print("T-bill shape:     ", tbill.shape)

C:\Users\sayan\AppData\Local\Temp\ipykernel_15544\2674284016.py:7: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  lendingclub = pd.read_csv(


LendingClub shape: (2260701, 5)
FED data shape:    (859, 2)
T-bill shape:      (240, 2)


## 2. Clean the LendingClub loan-level data

We do the following in this notebook:
- kept a small subset of columns,
- removed duplicate loan IDs,
- converted `issue_d` to a datetime,
- sorted by date,
- dropped rows with missing values.




In [3]:
# Remove duplicate loan IDs and keep the first occurrence.
duplicate_ids = lendingclub[lendingclub.duplicated(subset="id", keep=False)].sort_values("id")
lendingclub = lendingclub.drop_duplicates(subset="id", keep="first").copy()

# Keep the modeling columns and convert the issue date.
lendingclub = lendingclub[["issue_d", "loan_amnt", "term", "int_rate"]].copy()
lendingclub["issue_d"] = pd.to_datetime(lendingclub["issue_d"], format="%b-%Y")

# Sort chronologically and remove incomplete rows.
lendingclub = (
    lendingclub
    .sort_values("issue_d")
    .dropna()
    .reset_index(drop=True)
)

# Create a monthly period column for aggregation.
lendingclub["Month"] = lendingclub["issue_d"].dt.to_period("M")

print("Number of duplicated loan IDs removed:", len(duplicate_ids) // 2 if len(duplicate_ids) else 0)
print("Clean LendingClub shape:", lendingclub.shape)
lendingclub.head()

Number of duplicated loan IDs removed: 0
Clean LendingClub shape: (2260668, 5)


,issue_d,loan_amnt,term,int_rate,Month
0,2007-06-01,6500.0,36 months,8.38,2007-06
1,2007-06-01,6000.0,36 months,10.59,2007-06
2,2007-06-01,4400.0,36 months,9.64,2007-06
3,2007-06-01,1200.0,36 months,9.01,2007-06
4,2007-06-01,5000.0,36 months,11.22,2007-06


## 3. Aggregate LendingClub data to monthly level

This is the main transformation from loan-level observations to the monthly dataset used later for modeling.


In [4]:
lendingclub_monthly = (
    lendingclub
    .groupby("Month", as_index=False)
    .agg(
        int_rate_mean=("int_rate", "mean"),
        int_rate_median=("int_rate", "median"),
        int_rate_std=("int_rate", "std"),
        loan_amnt_sum=("loan_amnt", "sum"),
        loan_amnt_mean=("loan_amnt", "mean"),
        loan_amnt_count=("loan_amnt", "count"),
    )
    .sort_values("Month")
    .reset_index(drop=True)
)

print("Monthly LendingClub shape:", lendingclub_monthly.shape)
lendingclub_monthly.head()

Monthly LendingClub shape: (139, 7)


,Month,int_rate_mean,int_rate_median,int_rate_std,loan_amnt_sum,loan_amnt_mean,loan_amnt_count
0,2007-06,9.814583,9.640,1.886434,91850.0,3827.083333,24
1,2007-07,11.158571,10.590,2.830279,348325.0,5528.968254,63
2,2007-08,11.543514,11.065,3.116487,515300.0,6963.513514,74
3,2007-09,12.463208,11.860,3.157293,372950.0,7036.792453,53
4,2007-10,12.438476,12.490,2.698128,753225.0,7173.571429,105


## 4. Prepare the macroeconomic monthly series

We align the Treasury and Fed series to monthly periods,
then created a complete month range that starts **12 months before** the first LendingClub month and ends at the last LendingClub month.


In [5]:
def prepare_monthly_series(df, date_col, value_col, new_value_name):
    """Convert a dated series into a monthly Period-indexed two-column DataFrame."""
    out = df.rename(columns={date_col: "Date", value_col: new_value_name}).copy()
    out["Date"] = pd.to_datetime(out["Date"])
    out["Month"] = out["Date"].dt.to_period("M")
    return out[["Month", new_value_name]]

tbill_monthly = prepare_monthly_series(
    tbill,
    date_col="observation_date",
    value_col="TB3MS",
    new_value_name="Treasury_data",
)

fed_monthly = prepare_monthly_series(
    fed_data,
    date_col="observation_date",
    value_col="FEDFUNDS",
    new_value_name="fed_rate",
)

first_loan_month = lendingclub_monthly["Month"].min()
last_loan_month = lendingclub_monthly["Month"].max()

start_month = first_loan_month - 12
end_month = last_loan_month

combined_macro = pd.DataFrame(
    {"Month": pd.period_range(start=start_month, end=end_month, freq="M")}
)

combined_macro = (
    combined_macro
    .merge(tbill_monthly, on="Month", how="left")
    .merge(fed_monthly, on="Month", how="left")
    .sort_values("Month")
    .reset_index(drop=True)
)

print("Missing Treasury months:", combined_macro["Treasury_data"].isna().sum())
print("Missing Fed months:     ", combined_macro["fed_rate"].isna().sum())

combined_macro.head()

Missing Treasury months: 0
Missing Fed months:      0


,Month,Treasury_data,fed_rate
0,2006-06,4.79,4.99
1,2006-07,4.95,5.24
2,2006-08,4.96,5.25
3,2006-09,4.81,5.25
4,2006-10,4.92,5.25


## 5. Merge the monthly LendingClub data with the macro series

We perform a left join on `Month` so that every LendingClub month is retained.


In [6]:
lendingclub_fed_tbill = (
    lendingclub_monthly
    .merge(combined_macro, on="Month", how="left")
    .sort_values("Month")
    .reset_index(drop=True)
)

print("Final merged dataset shape:", lendingclub_fed_tbill.shape)
lendingclub_fed_tbill.head()

Final merged dataset shape: (139, 9)


,Month,int_rate_mean,int_rate_median,int_rate_std,loan_amnt_sum,loan_amnt_mean,loan_amnt_count,Treasury_data,fed_rate
0,2007-06,9.814583,9.640,1.886434,91850.0,3827.083333,24,4.61,5.25
1,2007-07,11.158571,10.590,2.830279,348325.0,5528.968254,63,4.82,5.26
2,2007-08,11.543514,11.065,3.116487,515300.0,6963.513514,74,4.20,5.02
3,2007-09,12.463208,11.860,3.157293,372950.0,7036.792453,53,3.89,4.94
4,2007-10,12.438476,12.490,2.698128,753225.0,7173.571429,105,3.90,4.76


## 6. Check for missing months in the final timeline

A continuous monthly timeline is important for time-series style modeling and train/test splitting.


In [7]:
start_period = lendingclub_fed_tbill["Month"].min()
end_period = lendingclub_fed_tbill["Month"].max()

full_range = pd.period_range(start=start_period, end=end_period, freq="M")
missing_months = full_range[~full_range.isin(lendingclub_fed_tbill["Month"])]

print(f"Number of missing months in final dataset: {len(missing_months)}")
print("Missing months:")
print(missing_months)

Number of missing months in final dataset: 0
Missing months:
PeriodIndex([], dtype='period[M]')


## 7. Create a chronological 75/25 train-test split

This is a **time-based split**, not a random split.

That matters because a random split would mix future months into the training data,
which would create leakage for forecasting or temporal modeling tasks.


In [8]:
n_rows = len(lendingclub_fed_tbill)
split_idx = int(n_rows * 0.75)

train = lendingclub_fed_tbill.iloc[:split_idx].copy()
test = lendingclub_fed_tbill.iloc[split_idx:].copy()

print("Train months:", train["Month"].min(), "->", train["Month"].max(), "| rows:", len(train))
print("Test months: ", test["Month"].min(), "->", test["Month"].max(), "| rows:", len(test))
print("Chronological overlap?", train["Month"].max() >= test["Month"].min())

Train months: 2007-06 -> 2016-01 | rows: 104
Test months:  2016-02 -> 2018-12 | rows: 35
Chronological overlap? False


## 8. Save the outputs

The final train and test sets are written to CSV files for downstream modeling.


In [ ]:
train.to_csv("../Data/train.csv", index=False)
test.to_csv("../Data/test.csv", index=False)

print("Saved files:")
print("- train.csv")
print("- test.csv")

Saved files:
- train.csv
- test.csv
